# 01 - Understanding fMRIPrep outputs

This tutorial explains the files produced by **fMRIPrep**, https://fmriprep.org/en/stable/

By the end, you should be able to:

- recognize the BIDS folder structure;
- understand common fMRIPrep file names and types;
- learn about some quality control steps to take to confirm output integrity;
- decide which file is appropriate for a downstream analysis.

> fMRIPrep produces several file types, plus metadata and quality-control reports. The right file you choose to take downstream all depends on the analysis you plan to perform.

## Before you begin

Set `derivatives_dir` below to the directory containing your fMRIPrep output. A typical layout is:

```text
derivatives/
└── fmriprep/
    ├── dataset_description.json
    ├── sub-01/
    │   ├── anat/
    │   └── func/
    └── sub-02/
```

For the purposes of this tutorial, I have compiled a derivative directory containing fMRIPrep outputs from the latest TAY data release, using fMRIPrep 25.2.3

In [ ]:
from pathlib import Path # this library is used to handle file paths in a way that works across different operating systems, mac, windows, linux.

import pandas as pd # pandas is a library that is used to handle csv, tsv, excel, and other data formats. It is extremely useful for data analysis and manipulation.

# path to the derivatives directory
derivatives_dir = Path("/projects/aabdulrasul/Tutorials/derivatives/fmriprep")

if derivatives_dir.exists():
    print(f"Using: {derivatives_dir.resolve()}")
else:
    print(f"No directory found at {derivatives_dir.resolve()}.")
    print("Update derivatives_dir when you are ready to inspect your own outputs.")

## 1. The BIDS-Derivatives layout

fMRIPrep follows the [BIDS-Derivatives](https://bids-specification.readthedocs.io/en/stable/derivatives/introduction.html) convention. The subject and session labels connect a derivative to its source scan:

```text
sub-01/
├── anat/
│   ├── sub-01_desc-preproc_T1w.nii.gz
│   ├── sub-01_from-MNI152NLin2009cAsym_to-T1w_mode-image_xfm.h5
│   └── sub-01_desc-aparcaseg_dseg.nii.gz
└── func/
    ├── sub-01_task-rest_run-01_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz
    ├── sub-01_task-rest_run-01_space-MNI152NLin2009cAsym_desc-confounds_timeseries.tsv
    └── sub-01_task-rest_run-01_space-MNI152NLin2009cAsym_desc-preproc_bold.json
```

The suffix tells you what the file contains (`T1w`, `bold`, `dseg`, or `timeseries`). Entities such as `task`, `run`, and `space` tell you which acquisition and coordinate system it belongs to.

In [ ]:
# List the subjects that fMRIPrep found.
subject_dirs = sorted(derivatives_dir.glob("sub-*")) if derivatives_dir.exists() else []

if subject_dirs:
    print(f"Found {len(subject_dirs)} subject(s):")
    for subject_dir in subject_dirs:
        print(f"  {subject_dir.name}")
else:
    print("No subject directories found yet. Check derivatives_dir above.")

## 2. Anatomical outputs: the subject's structural reference

The anatomical workflow uses a T1-weighted scan to build a subject-specific reference space.

- `desc-preproc_T1w.nii.gz`: bias-corrected, skull-stripped, and corrected T1w image.
- `desc-brain_mask.nii.gz`: binary mask of voxels considered part of the brain.
- `desc-aparcaseg_dseg.nii.gz`: discrete anatomical labels from the FreeSurfer-derived segmentation.
- `from-..._to-..._xfm.*` and `from-..._to-..._xfm.txt`: transforms between spaces.

These files are useful for displaying results on anatomy, defining regions, and moving data between native T1w and standard template spaces. The `space-MNI...` label means the image has been resampled into a template space; a file without a `space-` entity is often in the subject's native space.

In [ ]:
def show_matches(pattern: str, limit: int = 12):
    """Print a small sample of files matching a derivatives pattern."""
    matches = sorted(derivatives_dir.rglob(pattern)) if derivatives_dir.exists() else []
    for path in matches[:limit]:
        print(path)
    if len(matches) > limit:
        print(f"... and {len(matches) - limit} more")
    if not matches:
        print(f"No files matched {pattern!r}.")

print("Preprocessed anatomical images:")
show_matches("*desc-preproc_T1w.nii.gz")

print("\nBrain masks:")
show_matches("*desc-brain_mask.nii.gz")

print("\nSegmentations:")
show_matches("*dseg.nii.gz")

## 3. Functional outputs: the BOLD data

BOLD files are 4D NIfTI images: three spatial dimensions plus a time dimension (volumes or TRs).

The most common functional derivative is:

`sub-##_task-<name>_run-##_space-<space>_desc-preproc_bold.nii.gz`

`desc-preproc_bold` is motion-corrected and distortion-corrected BOLD data after fMRIPrep's preprocessing workflow. It is still **not automatically denoised**. Motion regressors, physiological estimates, and other nuisance variables are provided separately in the matching confounds TSV.

Use the JSON sidecar to inspect metadata such as repetition time (`RepetitionTime`) and the number of volumes. For group-level analyses, make sure that all subjects are represented in the same `space-` and use compatible preprocessing settings.

In [ ]:
print("Preprocessed BOLD images:")
show_matches("*desc-preproc_bold.nii.gz")

print("\nBOLD metadata sidecars:")
show_matches("*desc-preproc_bold.json")

## 4. Confounds: the nuisance information you may regress out

For each preprocessed BOLD run, fMRIPrep writes a matching `desc-confounds_timeseries.tsv`. Each row corresponds to one BOLD volume. Columns can include:

- head-motion estimates: `trans_*` and `rot_*`;
- framewise displacement: `framewise_displacement`;
- anatomical CompCor components: `a_comp_cor_*`;
- cosine drift regressors: `cosine*`;
- signals from masks such as `csf`, `white_matter`, and `global_signal`;
- non-steady-state volume indicators: `non_steady_state_outlier*`.

There is no universally correct confound set. Choose regressors based on your study design, then document that choice. Always check that the confounds TSV and BOLD NIfTI have the same number of rows/volumes before fitting a model.

In [ ]:
confounds_files = sorted(derivatives_dir.rglob("*desc-confounds_timeseries.tsv")) if derivatives_dir.exists() else []

if confounds_files:
    confounds_file = confounds_files[0]
    confounds = pd.read_csv(confounds_file, sep="\t")
    print(f"Example: {confounds_file}")
    print(f"Shape: {confounds.shape[0]} volumes x {confounds.shape[1]} confound columns")
    print("\nUseful columns present in this file:")
    keywords = ("motion", "trans_", "rot_", "framewise", "csf", "white_matter", "global_signal", "a_comp_cor")
    useful_columns = [column for column in confounds.columns if any(keyword in column for keyword in keywords)]
    print(useful_columns[:30])
else:
    print("No confounds TSV found yet. After setting derivatives_dir, rerun this cell.")

## 5. Quality control and provenance

Open the HTML report for each subject before trusting the derivatives. It summarizes registration, skull stripping, segmentation, susceptibility distortion correction, and functional preprocessing. Look for obvious failures such as poor brain extraction, misregistration, or large portions of signal outside the brain.

Other files help reproduce the workflow:

- `dataset_description.json` identifies the derivative dataset and pipeline version.
- `logs/` contains detailed execution logs when they were retained.
- `*.html` files contain visual QC reports.
- `*.json` sidecars describe acquisition and processing metadata.

A successful fMRIPrep command does not guarantee that every subject is suitable for every analysis. Visual QC and study-specific exclusion criteria still matter.

In [ ]:
print("Subject-level HTML reports:")
show_matches("*.html")

print("\nPipeline description files:")
show_matches("dataset_description.json")

## 6. Choosing a file for downstream work

Some examples:

| Goal | Start with |
| --- | --- |
| Inspect the subject's anatomy | `desc-preproc_T1w.nii.gz` and its brain mask |
| Run a task GLM | matching `desc-preproc_bold.nii.gz`, events from the raw BIDS dataset, and selected confounds |
| Denoise using established pipelines | XCP-D `bash_scripts/XCP_D_sbatch_tutorial.sh` |
| Analyze resting-state connectivity | matching preprocessed BOLD and a documented nuisance-regression strategy |
| Compare subjects in a common space | derivatives with the same `space-` label, commonly an MNI template |
| Check whether preprocessing worked | the subject HTML report and the BOLD reference image also the sbatch logs|
| Move data between spaces | the matching transform files |

